# Train TextCNN (Kaggle T4)

Notebook huan luyen TextCNN de phan loai `macro_domain` tu cau hoi phap luat.

In [1]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: Tesla T4


In [2]:
def detect_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]
    for p in candidates:
        if p.exists() and (p / 'qa_train.csv').exists():
            return p
    raise FileNotFoundError('Khong tim thay data_ready. Hay kiem tra duong dan dataset tren Kaggle.')

DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path('/kaggle/working/textcnn_artifacts' if Path('/kaggle').exists() else 'artifacts/textcnn')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

DATA_DIR = /kaggle/input/datasets/hngphtrn/legals/data_ready
ARTIFACT_DIR = /kaggle/working/textcnn_artifacts


In [3]:
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='\t')
qa_val = pd.read_csv(DATA_DIR / 'qa_val.csv', sep='\t')
qa_test = pd.read_csv(DATA_DIR / 'qa_test.csv', sep='\t')

required_cols = {'question', 'macro_domain'}
for name, df in [('train', qa_train), ('val', qa_val), ('test', qa_test)]:
    miss = required_cols - set(df.columns)
    if miss:
        raise ValueError(f'{name} missing columns: {miss}')

print('train:', qa_train.shape, 'val:', qa_val.shape, 'test:', qa_test.shape)
qa_train[['question','macro_domain']].head()

train: (23311, 14) val: (2841, 14) test: (2991, 14)


,question,macro_domain
0,"Theo Điều 5 Luật Địa chất và Khoáng sản, hội n...","Industry, Resources & Environment"
1,"Theo Điều 5 của Luật, khi thực hiện hội nhập v...","Industry, Resources & Environment"
2,Giả sử có một tranh chấp quốc tế phát sinh liê...,"Industry, Resources & Environment"
3,"Theo Điều 2 của Luật Địa chất và Khoáng sản, ""...","Industry, Resources & Environment"
4,Hãy phân tích sự khác biệt và mối quan hệ giữa...,"Industry, Resources & Environment"


In [4]:
# De co tokenizer tieng Viet tot hon, nen cai pyvi (on dinh hon tren Kaggle/Python 3.12).
!pip -q install pyvi
# Neu muon dung underthesea o moi truong khac:
# !pip -q install underthesea==6.8.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.6 MB/s eta 0:00:00


In [5]:
TOKENIZER_BACKEND = 'whitespace'
PYVI_IMPORT_ERROR = None
UNDERTHESEA_IMPORT_ERROR = None

# Uu tien pyvi (on dinh tren Kaggle/Python 3.12), neu khong co thi moi thu underthesea.
try:
    from pyvi import ViTokenizer
    TOKENIZER_BACKEND = 'pyvi'
except Exception as exc_pyvi:
    PYVI_IMPORT_ERROR = exc_pyvi
    try:
        from underthesea import word_tokenize
        TOKENIZER_BACKEND = 'underthesea'
    except Exception as exc_uts:
        UNDERTHESEA_IMPORT_ERROR = exc_uts
        TOKENIZER_BACKEND = 'whitespace'

print('Tokenizer backend:', TOKENIZER_BACKEND)
if TOKENIZER_BACKEND != 'pyvi' and PYVI_IMPORT_ERROR is not None:
    print('pyvi import error:', repr(PYVI_IMPORT_ERROR))
if TOKENIZER_BACKEND == 'whitespace' and UNDERTHESEA_IMPORT_ERROR is not None:
    print('underthesea import error:', repr(UNDERTHESEA_IMPORT_ERROR))


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize(text: str):
    text = normalize_text(text)
    if TOKENIZER_BACKEND == 'pyvi':
        return [t for t in ViTokenizer.tokenize(text).split() if t]
    if TOKENIZER_BACKEND == 'underthesea':
        return [t for t in word_tokenize(text, format='list') if t]
    return text.split()

Tokenizer backend: pyvi


In [6]:
PAD = '<PAD>'
UNK = '<UNK>'
MAX_VOCAB = 30000
MAX_LEN = 128

print("TOKENIZER_BACKEND=", TOKENIZER_BACKEND)
print("MAX_LEN=", MAX_LEN)

counter = Counter()
for q in qa_train['question'].astype(str).tolist():
    counter.update(tokenize(q))

vocab_tokens = [PAD, UNK] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(vocab_tokens)}
itos = {i: w for w, i in stoi.items()}

labels = sorted(qa_train['macro_domain'].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print('Vocab size:', len(stoi), '| Num labels:', len(labels))

TOKENIZER_BACKEND= pyvi
MAX_LEN= 128
Vocab size: 5582 | Num labels: 8


In [7]:
def encode_text(text, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [stoi.get(t, stoi[UNK]) for t in tokens][:max_len]
    if len(ids) < max_len:
        ids += [stoi[PAD]] * (max_len - len(ids))
    return ids

class QADomainDataset(Dataset):
    def __init__(self, df):
        self.questions = df['question'].astype(str).tolist()
        self.labels = [label2id[x] for x in df['macro_domain'].tolist()]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode_text(self.questions[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [8]:
BATCH_SIZE = 64

train_ds = QADomainDataset(qa_train)
val_ds = QADomainDataset(qa_val)
test_ds = QADomainDataset(qa_test)

train_label_ids = np.array([label2id[x] for x in qa_train['macro_domain'].tolist()])
class_counts = np.bincount(train_label_ids, minlength=len(labels))

from torch.utils.data import WeightedRandomSampler

# Oversample cac class yeu de tang xac suat model gap mau kho trong train.
# (Neu chi dung loss-weighting, co class van bi 'nuot' va recall khong nhay.)
class_sample_weights = 1.0 / (class_counts + 1e-6)
sample_weights = class_sample_weights[train_label_ids]

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))
print('Class counts:', class_counts.tolist())

23311 2841 2991
Class counts: [1167, 5580, 960, 3903, 2409, 888, 4515, 3889]


In [9]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes=(2,3,4), num_filters=128, dropout=0.6, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x)              # (B, L, D)
        emb = emb.transpose(1, 2)           # (B, D, L)
        feats = [F.relu(conv(emb)) for conv in self.convs]
        pooled = [F.max_pool1d(f, kernel_size=f.shape[2]).squeeze(2) for f in feats]
        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        return self.fc(out)

In [10]:
model = TextCNN(
    vocab_size=len(stoi),
    embed_dim=200,
    num_classes=len(labels),
    filter_sizes=(2,3,4),
    num_filters=128,
    dropout=0.6,
    pad_idx=stoi[PAD],
).to(device)

# Tang recall cho cac class yeu bang class-weighting manh hon + Focal Loss.
# (In thuc te: nhom recall thap thuong bi 'dai' vi loss chua tap trung vao mau kho.)
# Class weight manh hon cho class hiem.
# p = 1.0 (tức 1/freq) thuong tang recall nhung co the lam precision giam.
p = 1.0
class_weights = 1.0 / (class_counts + 1e-6) ** p
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

class FocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 3.0):
        super().__init__()
        self.register_buffer('class_weights', class_weights)
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()

        targets_log_probs = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        targets_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        at = self.class_weights.gather(0, targets)

        loss = -at * (1.0 - targets_probs).pow(self.gamma) * targets_log_probs
        return loss.mean()

criterion = FocalLoss(class_weights, gamma=3.0)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

In [11]:
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * x.size(0)

        # Inference-time logit adjustment (val/test only) de uuu tien lop hiem.
        # logits_for_pred = logits - log(p(class))
        logits_for_pred = logits
        if not train:
            log_prior = torch.log(
                torch.as_tensor(class_counts, device=logits.device, dtype=logits.dtype) + 1e-6
            )
            # Dieu chinh rieng cho lop 'Security & Defense' de tang recall.
            beta_security = 0.3
            sec_label = "Security & Defense"
            if sec_label in labels:
                sec_id = labels.index(sec_label)
                logits_for_pred = logits_for_pred.clone()
                logits_for_pred[:, sec_id] = logits_for_pred[:, sec_id] + beta_security * log_prior[sec_id]

        pred = torch.argmax(logits_for_pred, dim=1)
        y_true.extend(y.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    return avg_loss, macro_f1, y_true, y_pred

In [12]:
EPOCHS = 20
PATIENCE = 5
best_val_f1 = -1.0
wait = 0
best_path = ARTIFACT_DIR / 'textcnn_best.pt'

history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_f1, _, _ = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(va_f1)

    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_f1': tr_f1, 'val_loss': va_loss, 'val_f1': va_f1})
    print(f'Epoch {epoch:02d} | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | val_loss={va_loss:.4f} val_f1={va_f1:.4f}')

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        wait = 0
        torch.save(model.state_dict(), best_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping triggered.')
            break

print('Best val macro F1:', round(best_val_f1, 4))

Epoch 01 | train_loss=0.5522 train_f1=0.5368 | val_loss=0.4572 val_f1=0.4244
Epoch 02 | train_loss=0.1338 train_f1=0.8488 | val_loss=0.5330 val_f1=0.5314
Epoch 03 | train_loss=0.0723 train_f1=0.9071 | val_loss=0.5842 val_f1=0.5359
Epoch 04 | train_loss=0.0565 train_f1=0.9260 | val_loss=0.5602 val_f1=0.5734
Epoch 05 | train_loss=0.0428 train_f1=0.9419 | val_loss=0.6434 val_f1=0.5625
Epoch 06 | train_loss=0.0377 train_f1=0.9492 | val_loss=0.7398 val_f1=0.5584
Epoch 07 | train_loss=0.0296 train_f1=0.9575 | val_loss=0.7034 val_f1=0.5558
Epoch 08 | train_loss=0.0278 train_f1=0.9595 | val_loss=0.7761 val_f1=0.5632
Epoch 09 | train_loss=0.0240 train_f1=0.9631 | val_loss=0.7437 val_f1=0.5607
Early stopping triggered.
Best val macro F1: 0.5734


In [13]:
model.load_state_dict(torch.load(best_path, map_location=device))
te_loss, te_f1, y_true, y_pred = run_epoch(model, test_loader, optimizer=None)
print('Test loss:', round(te_loss, 4), '| Test macro F1:', round(te_f1, 4))
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))

Test loss: 0.8286 | Test macro F1: 0.6169
                                   precision    recall  f1-score   support

               Civil & Investment     0.1600    0.0242    0.0421       165
                Finance & Banking     0.9930    0.5776    0.7304       741
Industry, Resources & Environment     0.9927    0.6570    0.7907       207
     Justice & Dispute Resolution     0.7782    0.9479    0.8547       729
                Labor & Insurance     0.7500    0.6667    0.7059        18
               Security & Defense     0.1373    0.2838    0.1850       222
       State Organization & Admin     0.6263    0.7652    0.6888       609
                   Transportation     0.9519    0.9233    0.9374       300

                         accuracy                         0.6944      2991
                        macro avg     0.6737    0.6057    0.6169      2991
                     weighted avg     0.7509    0.6944    0.6986      2991



In [14]:
metadata_out = {
    'version': 'textcnn_v2',
    'tokenizer_backend': TOKENIZER_BACKEND,
    'max_len': MAX_LEN,
    'labels': labels,
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'pad_token': PAD,
    'unk_token': UNK,
    'train_strategy': {
        'sampling': 'shuffle',
        'class_weighting': 'inv_freq_pow_1.0',
        'label_smoothing': 0.0,
        'lr': 5e-4,
        'weight_decay': 0.0,
        'dropout': 0.6,
    }
}
with open(ARTIFACT_DIR / 'tokenizer_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)
with open(ARTIFACT_DIR / 'textcnn_meta.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_out, f, ensure_ascii=False, indent=2)

pd.DataFrame(history).to_csv(ARTIFACT_DIR / 'train_history.csv', index=False)
print('Saved artifacts at:', ARTIFACT_DIR)

Saved artifacts at: /kaggle/working/textcnn_artifacts
